# Exercise 3 - LLM Inference
_By Georg Ahnert_

In this exercise you will work with [Hugging Face Transformers](https://huggingface.co/docs/transformers/index) and [vllm](https://docs.vllm.ai/en/stable/index.html) to use generative language models for text annotation. We will focus on open-weight models that can be run locally (if you have a GPU available).

Note that the models we will use in this exercise are significantly larger (>1B parameters) than, e.g., distilbert (67M parameters). <br>
**This means that you will likely need to run this notebook on the BWUniCluster3.0 or on Google Colab to have enough GPU memory and compute.**

This exercise consists of five parts:
1. Running generative models with Hugging Face Transformers
2. Running instruction-tuned (chat) models with Hugging Face Transformers
3. Running instruction-tuned models with vllm
5. Structured outputs

### Setup


#### Only on BWUniCluster3.0
If you run this notebook for the first time on the BWUniCluster3.0, you'll have to do some initial setup.

First, make sure that you have started a `minimal` jupyter instance as **starting an `ai` instance will likely lead to package conflicts!** Once the instance is started, open a new Terminal tab. Then, create a virtual environment (similar to a conda environment) and activate the environment like this:

```bash
python -m venv llm4ess_env
source llm4ess_env/bin/activate
```

Next, we need to install the minimum requirements for using our new environment as a kernel for Jupyter Notebooks. And finally, we can create a kernel based on this environment:

```bash
pip install ipykernel ipywidgets
python -m ipykernel install --user --name llm4ess --display-name "Python (LLM4ESS)" 
```

Now you can close the terminal and reload the browser window. The new kernel named `Python (LLM4ESS)` should now show up in the drop-down menu on the top right of your jupyter notebook.

#### For both Google Colab and BWUniCluster3.0
Next, we have to install some additional dependencies - this step is required on both Google Colab and on the BWUniCluster and might take a while...

In [ ]:
%%capture
%pip install vllm # note that vllm also installs many dependencies such as transformers, torch, pydantic etc.
%pip install seaborn

## Part 1: Running Generative Models with 🤗 Transformers

First, let's import the required classes from Hugging Face Transformers and initialize our model.

Here, we use [allenai/OLMo-2-0425-1B](https://huggingface.co/allenai/OLMo-2-0425-1B), an open language model from AI2. We can simply initialize it by its Hugging Face model name and the transformers library will take care of the rest (downloading the model, etc.).

Note that allenai/OLMo-2-0425-1B is a base model, i.e., it is trained to complete text input, not to engage in a dialog with the user. We'll have a look at instruction-tuned models later.

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, set_seed
import torch

set_seed(42) # set a fixed seed for reproducibility

tokenizer = AutoTokenizer.from_pretrained("allenai/OLMo-2-0425-1B")
model = AutoModelForCausalLM.from_pretrained("allenai/OLMo-2-0425-1B")

model = model.to('cuda') # load the model onto the GPU

We should now have the model successfully loaded in the GPU – let's have a look…

In [ ]:
!nvidia-smi  # this is a command line tool that gives you an overview over your current GPU usage

To make use of our language model, we need to provide it with some text input. Let's convert some texts into tokens…

In [ ]:
message = ["The University of Mannheim is "]
inputs = tokenizer(message, return_tensors='pt', return_token_type_ids=False)
inputs

After moving our tokens to the GPU as well, we can generate text completions.
Note that is is important to have both the model and the input tokens on the same device.

In [ ]:
inputs = {k: v.to('cuda') for k,v in inputs.items()}

response = model.generate(**inputs, max_new_tokens=100, do_sample=True, top_k=50, top_p=0.95)
response

Finally, we can de-code the tokens that we got in response and examine the text. It's probably not very impressive yet, though.

In [ ]:
print(tokenizer.batch_decode(response, skip_special_tokens=True)[0])

Let's try out some other generation parameters:

In [ ]:
print('--- greedy decoding ---')
response = model.generate(**inputs, max_new_tokens=100, do_sample=False)
print(tokenizer.batch_decode(response, skip_special_tokens=True)[0])

print('\n--- temperature 0.1 ---')
response = model.generate(**inputs, max_new_tokens=100, do_sample=True, temperature=0.1)
print(tokenizer.batch_decode(response, skip_special_tokens=True)[0])

print('\n--- temperature 1.0 ---')
response = model.generate(**inputs, max_new_tokens=100, do_sample=True, temperature=1.0)
print(tokenizer.batch_decode(response, skip_special_tokens=True)[0])

print('\n--- temperature 2.0 ---')
response = model.generate(**inputs, max_new_tokens=100, do_sample=True, temperature=2.0)
print(tokenizer.batch_decode(response, skip_special_tokens=True)[0])

We can also investigate the probabilities of each token in the vocabulary at each step of the output:

In [ ]:
response = model.generate(**inputs, max_new_tokens=100, do_sample=False, return_dict_in_generate=True, output_logits=True)
response.logits[0][0] # first token in output and first output in batch

In [ ]:
# response.logits has shape: (seq_len, batch_size, vocab_size)
for step, step_logits in enumerate(response.logits[:5], start=1):

    print(f"--- Step {step} ---")
    
    log_probs = torch.nn.functional.softmax(step_logits[0], dim=-1) # apply softmax to convert to probabilities
    
    topk = torch.topk(log_probs, k=10) # extract the 10 most likely tokens at each step
    top_ids = topk.indices.tolist()
    top_scores = topk.values.tolist()

    decoded_tokens = tokenizer.batch_decode(top_ids)
    for tok, score in zip(decoded_tokens, top_scores):
        print(f"'{tok}': {score:.3f}", end=', ')
    
    print('\n')


### Task 1: Try out the impact of temperature and top_k in combination. What do you observe in the generated text?

## Part 2: Running Chat Models with 🤗 Transformers

    Important: Restart your Python kernel now to clear the GPU memory

The base version of OLMo 1B wasn't exactly helpful, because we couldn't easily ask a question. Let's try out the instruction-tuned version instead.

In [ ]:
from transformers import pipeline

# pipelines are a higher-level abstraction offered by the transformers library
# they can be used with both base models and instruction-tuned models
pipe = pipeline("text-generation", model="allenai/OLMo-2-0425-1B-Instruct")

# also notice how the device was automatically set to 'cuda', i.e., the GPU

messages = [{"role": "user", "content": "Who are you?"}]
pipe(messages)

Great, we have a working chat model now that we can ask to answer our questions or to perform tasks such as text annotation. We can use messages with different roles for system, and user prompts.

In [ ]:
messages = [
    {"role": "system", "content": "You are a pirate."},
    {"role": "user", "content": "Describe yourself"}
]
pipe(messages)[0]['generated_text']

We can also pre-define previous `assistant` answers. This can be helpful for few-shot prompting – notice how the LLM adopts our answer template in the following example:

In [ ]:
messages = [
    {"role": "system", "content": "You are an expert mathematician."},
    {"role": "user", "content": "What is 4+4?"},
    {"role": "assistant", "content": "The answer is 8."},
    {"role": "user", "content": "What is 49/7?"}
]
pipe(messages)[0]['generated_text']

Under the hood, the model still generates tokens first, which are then automatically decoded and parsed into a list of dictionaries. Notice how the output contains special tokens such as `<|user|>` and `<|assistant|>` that allow us to differentiate between texts from different sources.

In [ ]:
result = pipe(messages, return_tensors=True)

print('--- first 10 output tokens ---')
print(result[0]['generated_token_ids'][:10])

print('\n--- decoded output text ---')
tokenizer = pipe.tokenizer
print(''.join(tokenizer.batch_decode(result[0]['generated_token_ids'])))

We'll have a look at prompting in more detail next week.

### Task 2: Test OLMo's mathematical skills

Using a pipeline like the one above, investigate which maths problems OLMo 1B can solve correctly and which it can't.

## Part 3: Running instruction-tuned models with vllm

While the 🤗 Transformers library makes it relatively easy to use (chat) models, processing lots of text with it is not very computationally efficient. To demonstrate this, let's run a very simple benchmark:

In [ ]:
%%time
messages = []
for number in range(100):
    messages.append([
        {"role": "system", "content": "You are an expert mathematician."},
        {"role": "user", "content": "What is 4+4?"},
        {"role": "assistant", "content": "The answer is 8."},
        {"role": "user", "content": f"What is 49/{number}?"}
    ])

result = pipe(messages, max_new_tokens=10)

[vllm](https://docs.vllm.ai/en/latest/getting_started/quickstart.html) is a library that is built for very efficient model inference. It also allows you to process multiple inputs in batches without having to worry about GPU memory overflow.

To start OLMo 1B with vllm, first `restart the Python kernel (again) to clear the GPU memory`. Then, load the model with vllm as follows:

In [ ]:
from vllm import LLM, SamplingParams

llm = LLM("allenai/OLMo-2-0425-1B-Instruct")

A couple of things to note:
- vllm uses the same **huggingface model cache**, so we didn't have to download OLMo 1B again
- vllm uses many optimizations like Flash Attention or torch.compile to speed up inference
- vllm also uses **automatic prefix-caching**, so the same prompt prefix only has to be processed once

On GPU memory utilization:
- vllm already reserves 90 % of the GPU memory for batch processing, even if the model itself is much smaller - here, this results in a maximum concurrency of ~13x
- if you have much smaller request, it makes sense to reduce the context length with `max_model_len` (4096 tokens by default for OLMo), enabling larger batch sizes or allowing you to run larger models

Further options for reducing memory usage - see also the [vllm documentation](https://docs.vllm.ai/en/stable/configuration/conserving_memory.html) for more information:
- set `gpu_memory_utilization` to a value > 0.9
- disable torch.compile with `enforce_eager = True` - but this results in slower processing
- use `quantization` - but this will decrease model performance for some tasks
- use multiple GPUs with `tensor_parallel_size` ≥2

Ok, let's run our simple benchmark again:

In [ ]:
messages = []

for number in range(100):
    messages.append([
        {"role": "system", "content": "You are an expert mathematician."},
        {"role": "user", "content": "What is 4+4?"},
        {"role": "assistant", "content": "The answer is 8."},
        {"role": "user", "content": f"What is 49/{number}?"}
    ])

sampling_params = SamplingParams(max_tokens=10)

In [ ]:
%%time
outputs = llm.chat(messages, sampling_params=sampling_params)

vllm's `llm.chat()` function works very similar to the huggingface pipeline. One major difference is that we have to provide the parameters for text generation now as a `SamplingParams` object.

### Task 3a: Generate random integers between 1 and 100 using OLMo 1B and vllm. Visualize the distribution of generated numbers.

### Task 3b: What happens if you use a low temperature?

## Part 4: Structured Outputs

As you might have notices in Task 3, language models might not always adhere to the formatting instructions that we provide. This is especially problematic if we have a more complicated output structure such as the following JSON:

```json
{
    "id": <an integer>,
    "first name": <a string>,
    "email": <a valid email address>,
    "gender": <'female'/'male'/'non-binary'>
}
```

In [ ]:
json_format = """
{
    "id": <an integer>,
    "first_name": <a string>,
    "email": <a valid email address>,
    "gender": <"female"/"male"/"non-binary">
}
"""

messages = [
    {"role": "system", "content": "You are a helpful assistant generating database entries in the following JSON format:" + json_format},
    {"role": "user", "content": "Generate an entry for Maxi Mustermann."}
]

sampling_params = SamplingParams(max_tokens=200, temperature=1.2, seed=2)
outputs = llm.chat(messages, sampling_params=sampling_params)

In [ ]:
print(outputs[0].outputs[0].text)

Even if OLMo might have tried to provide some useful context here, we would not be able to directly parse the generated text with `json.loads()`. However, we can **force vllm to adhere to the desired output format by restricting the model's vocabulary** at each step of the text generation. This is called *Structured Outputs* - see also the [vllm documentation](https://docs.vllm.ai/en/stable/features/structured_outputs.html).

With vllm, we can for instance restrict the output to a choice from a list of answer options:

In [ ]:
from vllm.sampling_params import GuidedDecodingParams

guided_decoding_params = GuidedDecodingParams(choice=["Positive", "Negative"])
sampling_params = SamplingParams(guided_decoding=guided_decoding_params, seed=1)

outputs = llm.generate(
    prompts="Classify this sentiment: Mannheim is wonderful!",
    sampling_params=sampling_params,
)
print(outputs[0].outputs[0].text)

To get a more elaborate JSON output pattern, we can define the desired output as a class using [PyDantic](https://docs.pydantic.dev/latest/):

In [ ]:
from pydantic import BaseModel, EmailStr
from enum import Enum

class GenderType(str, Enum):
    female = "female"
    male = "male"
    non_binary = "non-binary"

class DatabaseEntry(BaseModel):
    id: int
    first_name: str
    email: EmailStr # a special type for validating email addresses
    gender: GenderType # one of the choices from the enum above

In [ ]:
json_format = """
{
    "id": <an integer>,
    "first_name": <a string>,
    "email": <a valid email address>,
    "gender": <"female"/"male"/"non-binary">
}
"""

messages = [
    {"role": "system", "content": "You are a helpful assistant generating database entries in the following JSON format:" + json_format},
    {"role": "user", "content": "Generate an entry for Maxi Mustermann."}
]

guided_decoding_params = GuidedDecodingParams(json=DatabaseEntry.model_json_schema())
sampling_params = SamplingParams(max_tokens=200, temperature=1.2, seed=2, guided_decoding=guided_decoding_params)
outputs = llm.chat(messages, sampling_params=sampling_params)

In [ ]:
print(outputs[0].outputs[0].text)

Note that most other inference providers (such as the OpenAI API) **only offer structured outputs with json schema.**

Also note that the **model is not "aware" of the pattern**, so you should still write an appropriate formatting instruction in the system/user prompt and just use structured outputs additionally.

### Task 4: Ask OLMo for a Verbalized Distribution of Survey Answers

Simulate survey responses for Germany on question Q113 of the [World Value Survey (wave 7)](https://www.worldvaluessurvey.org/WVSDocumentationWV7.jsp). Use structured outputs to obtain probabilities for all possible answer options.